# Aegis — QLoRA training on Qwen 2.5 7B-Instruct

Run on Colab Pro (A100 40GB) connected via VS Code's Jupyter extension.
Cells run top-to-bottom. Toggle `DRY_RUN = True` in the Configuration cell
to verify the pipeline on 50 steps × 100 examples (~3-5 min on A100) before
committing to the real ~3-5 hour run.

**Hyperparameters** (locked to the project spec):
- Base: Qwen 2.5 7B-Instruct, 4-bit
- LoRA: r=16, α=32, all linear layers
- lr=2e-4, batch=4 × grad_accum=4, 2 epochs, cosine, 5% warmup, seed=42

**Outputs:**
- Checkpoints to `/content/drive/MyDrive/aegis-checkpoints/<run_name>/`
- Best-by-`eval_loss` restored at end
- Final adapter pushed to HF Hub; periodic pushes every 4 hours during training

## 1. Setup

Clone the repo, install the training stack, set env vars, mount Drive,
generate the processed dataset.

In [7]:
import os

REPO_URL = "https://github.com/Venkata1345/Aegis-finetuning.git" 
REPO_DIR = "/content/fine-tuning"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Cloning into '/content/fine-tuning'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 52 (delta 1), reused 52 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 102.16 KiB | 10.22 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/fine-tuning


In [8]:
!pip install -q -r requirements.txt
!pip install -q -r requirements-train.txt
!pip install -q -e .

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 127.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 13.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 47.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 47.0.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing

In [1]:
# Fill in your real values; do NOT commit them anywhere.
import os
os.environ["HF_TOKEN"]       = "hf_..."
os.environ["WANDB_API_KEY"]  = "..."
os.environ["AEGIS_HUB_REPO"] = "Abhishek4545/aegis-qwen-7b-lora"
os.environ["AEGIS_RUN_NAME"] = "aegis-qwen-7b-v1"
os.environ["WANDB_PROJECT"]  = "aegis" 

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# One-time per run. Skip if data/processed/train.chat.jsonl already exists.
!python -m data.download
!python -m data.convert
!python -m data.format

Loading ai4privacy/pii-masking-300k from Hugging Face Hub...
README.md: 16.7kB [00:00, 18.0MB/s]
data/train/1english_openpii_30k.jsonl: 100% 103M/103M [00:02<00:00, 36.3MB/s]
data/train/dutch_openpii_28k.jsonl: 100% 102M/102M [00:02<00:00, 50.9MB/s]
data/train/french_openpii_31k.jsonl: 100% 114M/114M [00:01<00:00, 94.0MB/s]
data/train/german_openpii_30k.jsonl: 100% 108M/108M [00:01<00:00, 59.4MB/s]
data/train/italian_openpii_29k.jsonl: 100% 104M/104M [00:01<00:00, 57.3MB/s]
data/train/spanish_openpii_29k.jsonl: 100% 102M/102M [00:02<00:00, 46.0MB/s]
data/validation/1english_openpii_8k.json(…): 100% 27.3M/27.3M [00:00<00:00, 44.7MB/s]
data/validation/dutch_openpii_7k.jsonl: 100% 27.0M/27.0M [00:00<00:00, 65.6MB/s]
data/validation/french_openpii_8k.jsonl: 100% 30.7M/30.7M [00:01<00:00, 19.0MB/s]
data/validation/german_openpii_8k.jsonl: 100% 29.2M/29.2M [00:01<00:00, 20.7MB/s]
data/validation/italian_openpiii_8k.json(…): 100% 28.3M/28.3M [00:00<00:00, 68.9MB/s]
data/validation/spanish_ope

## 2. Configuration

Hyperparameters and run paths. Toggle `DRY_RUN = True` to verify the pipeline
on 50 steps × 100 examples before the real run.

In [3]:
import os
from datetime import datetime
from pathlib import Path

# Toggle for smoke test (50 steps × 100 examples, ~3-5 min on A100)
DRY_RUN = False

MODEL_ID = "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048

LORA_R, LORA_ALPHA = 16, 32
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

LR             = 2e-4
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
NUM_EPOCHS     = 2
WARMUP_RATIO   = 0.05
SAVE_STEPS     = 500
EVAL_STEPS     = 500
LOGGING_STEPS  = 10
HUB_PUSH_HOURS = 4
SEED           = 42

DRY_RUN_STEPS = 50
DRY_RUN_TRAIN = 100
DRY_RUN_VAL   = 50

DATA_DIR        = Path("/content/fine-tuning/data/processed")
DRIVE_CKPT_ROOT = Path("/content/drive/MyDrive/aegis-checkpoints")
DRIVE_CKPT_ROOT.mkdir(parents=True, exist_ok=True)

run_name = (os.environ.get("AEGIS_RUN_NAME")
            or datetime.now().strftime("aegis-%Y%m%d-%H%M%S"))
if DRY_RUN:
    run_name = f"{run_name}-dryrun"
RUN_DIR = DRIVE_CKPT_ROOT / run_name
RUN_DIR.mkdir(exist_ok=True)
print(f"Run name:      {run_name}")
print(f"Run directory: {RUN_DIR}")
print(f"DRY_RUN:       {DRY_RUN}")

Run name:      aegis-qwen-7b-v1
Run directory: /content/drive/MyDrive/aegis-checkpoints/aegis-qwen-7b-v1
DRY_RUN:       False


In [13]:
!pip uninstall -y unsloth unsloth_zoo
!pip install --no-cache-dir "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --no-cache-dir "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip show unsloth unsloth_zoo | grep -E "Name|Version"


Found existing installation: unsloth 2026.4.8
Uninstalling unsloth-2026.4.8:
  Successfully uninstalled unsloth-2026.4.8
Found existing installation: unsloth_zoo 2026.4.9
Uninstalling unsloth_zoo-2026.4.9:
  Successfully uninstalled unsloth_zoo-2026.4.9
  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-install-4swen8ud/unsloth-zoo_419064358a054c07b16e8c06c54e6114
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-install-4swen8ud/unsloth-zoo_419064358a054c07b16e8c06c54e6114
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit 1894d8815657146cd2f311fa160d250bf8b8f2c2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth_zoo: filename=unsloth_zoo-2026.4.9-py3-none-any.whl size=424450 sha256=ee63ca85331c546e0b308d64fa45f33e765dd5c271affb4ee94e0fb76c9113c4
  Stored in directory: /tmp/pip-ephem-wh

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ygmduo6v/unsloth_9e0200f164bf4794abf24f507c73266b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ygmduo6v/unsloth_9e0200f164bf4794abf24f507c73266b
  Resolved https://github.com/unslothai/unsloth.git to commit 5533bdb8b63ecfef2685ef67892ead83f8c54fb3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.8-py3-none-any.whl size=32230676 sha256=ca55548eff4741ab58dcdcbb3a03246e02c1f938d2097e4693ac87faa5a7b85e
  Stored in directory: /tmp/pip-ephem-wheel-cache-6dfjaj6k/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
Name: unsloth
Version: 2026.4.8
Name: unsloth_zoo
Version: 2026.4.9


In [4]:
!pip install -U "bitsandbytes>=0.46.1"


## 3. Load model

Loads Qwen 2.5 7B-Instruct in 4-bit via Unsloth and wraps it with LoRA.
First run downloads ~5 GB of weights — takes a few minutes.

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")


torch: 2.10.0+cu128
cuda available: True
device name: NVIDIA A100-SXM4-80GB


## 4. Load and format data

Reads chat-format JSONL and applies the Qwen chat template.

In [7]:
import json
from datasets import Dataset


def load_chat_jsonl(path: Path, limit: int | None = None) -> Dataset:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return Dataset.from_list(rows)


train_limit = DRY_RUN_TRAIN if DRY_RUN else None
val_limit   = DRY_RUN_VAL   if DRY_RUN else None

train_ds = load_chat_jsonl(DATA_DIR / "train.chat.jsonl", limit=train_limit)
val_ds   = load_chat_jsonl(DATA_DIR / "val.chat.jsonl",   limit=val_limit)
print(f"train: {len(train_ds)}  val: {len(val_ds)}")


def apply_chat_template(batch):
    return {"text": [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch["messages"]
    ]}


train_ds = train_ds.map(apply_chat_template, batched=True, remove_columns=["messages"])
val_ds   = val_ds.map(apply_chat_template,   batched=True, remove_columns=["messages"])

train: 28412  val: 1496


Map:   0%|          | 0/28412 [00:00<?, ? examples/s]

Map:   0%|          | 0/1496 [00:00<?, ? examples/s]

## 5. Train

Configure SFTTrainer and start training. Set `RESUME = True` in the train cell
to pick up the latest checkpoint in `RUN_DIR`.

In [8]:
import time
from trl import SFTConfig, SFTTrainer
from transformers import TrainerCallback


class PeriodicHubPush(TrainerCallback):
    """Push the LoRA adapter to HF Hub every `every_hours` hours."""

    def __init__(self, repo_id: str, every_hours: float):
        self.repo_id = repo_id
        self.interval_seconds = every_hours * 3600
        self.last_push = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if now - self.last_push < self.interval_seconds:
            return
        m = kwargs.get("model")
        if m is None:
            return
        print(f"\n[step {state.global_step}] periodic push -> {self.repo_id}")
        try:
            m.push_to_hub(self.repo_id,
                          commit_message=f"checkpoint at step {state.global_step}")
        except Exception as e:
            print(f"push FAILED: {e}")
        self.last_push = now


eval_steps = EVAL_STEPS if not DRY_RUN else max(1, DRY_RUN_STEPS // 2)
save_steps = SAVE_STEPS if not DRY_RUN else max(1, DRY_RUN_STEPS // 2)
max_steps  = DRY_RUN_STEPS if DRY_RUN else -1
epochs     = 1 if DRY_RUN else NUM_EPOCHS

sft_config = SFTConfig(
    output_dir=str(RUN_DIR),
    num_train_epochs=epochs,
    max_steps=max_steps,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=eval_steps,
    logging_steps=LOGGING_STEPS,
    report_to=["wandb"] if os.environ.get("WANDB_API_KEY") else [],
    run_name=run_name,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    push_to_hub=False,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    seed=SEED,
    data_seed=SEED,
)

callbacks = []
hub_repo = os.environ.get("AEGIS_HUB_REPO")
if hub_repo and not DRY_RUN:
    callbacks.append(PeriodicHubPush(hub_repo, HUB_PUSH_HOURS))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    callbacks=callbacks,
)
print("Trainer ready.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/28412 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1496 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer ready.


In [9]:
RESUME = True  # set True to resume from latest checkpoint in RUN_DIR

resume_path = None
if RESUME:
    candidates = [d for d in RUN_DIR.iterdir()
                  if d.is_dir() and d.name.startswith("checkpoint-")]
    if candidates:
        latest = max(candidates, key=lambda d: int(d.name.split("-")[-1]))
        resume_path = str(latest)
        print(f"Resuming from {latest}")
    else:
        print("No checkpoint found; starting fresh.")

print(f"Starting training at {datetime.now().isoformat(timespec='seconds')}")
trainer.train(resume_from_checkpoint=resume_path)
print(f"Training complete at {datetime.now().isoformat(timespec='seconds')}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming from /content/drive/MyDrive/aegis-checkpoints/aegis-qwen-7b-v1/checkpoint-1000
Starting training at 2026-05-04T20:25:50


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,412 | Num Epochs = 2 | Total steps = 3,552
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: venkataabhishek054 (venkataabhishek054-ecare-medical-group) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 49.4kB /  162MB            

Saved model to https://huggingface.co/Abhishek4545/aegis-qwen-7b-lora


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/aegis-checkpoints/aegis-qwen-7b-v1/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/aegis-checkpoints/aegis-qwen-7b-v1/checkpoint-3500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/aegis-checkpoints/aegis-qwen-7b-v1/checkpoint-3552/tokenizer_config.json.


Training complete at 2026-05-05T02:06:56


## 6. Push final adapter

Saves the LoRA adapter to your HF Hub repo. Pull it locally for eval via:

```bash
huggingface-cli download <your-username>/aegis-qwen-7b-lora --local-dir adapters/aegis
```

In [10]:
if hub_repo and not DRY_RUN:
    print(f"Pushing final adapter to {hub_repo}...")
    model.push_to_hub(hub_repo, commit_message="final adapter")
    tokenizer.push_to_hub(hub_repo)
    print(f"Pushed: https://huggingface.co/{hub_repo}")
else:
    print("Skipped Hub push (DRY_RUN active or AEGIS_HUB_REPO not set).")
print(f"\nFinal artifact: {RUN_DIR}")

Pushing final adapter to Abhishek4545/aegis-qwen-7b-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 49.4kB /  162MB            

Saved model to https://huggingface.co/Abhishek4545/aegis-qwen-7b-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp9kzqpyiv/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9kzqpyiv/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Pushed: https://huggingface.co/Abhishek4545/aegis-qwen-7b-lora

Final artifact: /content/drive/MyDrive/aegis-checkpoints/aegis-qwen-7b-v1


## Done

After training:
1. Pull the adapter locally with `huggingface-cli download`.
2. Run `python -m eval.run_baseline aegis` (Aegis predictor — to be added).
3. Run `python -m eval.harness --inject README.md` to update the comparison table.

The delta between the Presidio / GPT-4o-mini / Gemini Flash baselines
(already in the README) and the Aegis numbers is the project's headline result.